# Africa-USA Road Infrastructure & Cross-Border Trade Corridors — Analysis

**Domain:** Global Transportation  |  **Scope:** 15 African countries + USA benchmark  |  **Source:** World Bank Indicators API

This notebook explores the SQLite warehouse produced by the pipeline (`src/ingestion` → `src/transformation` → `src/loading`).
It asks four questions:

1. How far behind the US logistics benchmark are Africa's main trade-corridor economies?
2. What does the physical road network (density and paving) look like across the region?
3. Does a denser road network go hand in hand with greater trade openness?
4. How large are African merchandise exports relative to the USA, and who dominates?

> **Data caveats.** The World Bank files the 2023 LPI edition under year **2022**. Road density and paved-road
> series are legacy indicators last updated between 2001 and 2010, so they describe the *structural* network and are
> compared with recent (2021-2025) trade data. Nigeria has no trade-%-of-GDP value in the extract.

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", "{:,.2f}".format)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = ROOT / "data" / "transport_trade.db"
assert DB_PATH.exists(), f"{DB_PATH} missing - run the pipeline first (ingest -> transform -> load)"

with sqlite3.connect(DB_PATH) as conn:
    summary = pd.read_sql("SELECT * FROM country_summary", conn)
    lpi = pd.read_sql("SELECT * FROM lpi_rankings", conn)
    master = pd.read_sql("SELECT * FROM master_transport_trade", conn)
    raw = pd.read_sql("SELECT * FROM raw_indicators", conn)

SHORT = {"Egypt, Arab Rep.": "Egypt", "Cote d'Ivoire": "Côte d'Ivoire", "United States": "USA"}
for df in (summary, lpi, master, raw):
    df["label"] = df["country_name"].replace(SHORT)

print(f"country_summary: {summary.shape}, lpi_rankings: {lpi.shape}, master: {master.shape}, raw: {raw.shape}")
summary[["label", "region", "lpi_overall", "lpi_overall_year", "lpi_gap_vs_usa",
         "trade_pct_gdp", "trade_openness_category", "lpi_rank"]]

## 1. Africa vs USA — Logistics Performance Index

The LPI (1 = low, 5 = high) measures customs, infrastructure, shipments, logistics competence, tracking and timeliness.
Each country is shown with its **latest available** edition, against the US score.

In [ ]:
plot_df = summary.sort_values("lpi_overall", ascending=True)
colors = ["#1f4e79" if r == "North America" else "#e07b39" for r in plot_df["region"]]
usa_lpi = summary.loc[summary["country_code"] == "USA", "lpi_overall"].iloc[0]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(plot_df["label"], plot_df["lpi_overall"], color=colors)
ax.axvline(usa_lpi, color="#1f4e79", ls="--", lw=1.2, label=f"USA benchmark ({usa_lpi:.2f})")
africa_avg = summary.loc[summary["region"] == "Africa", "lpi_overall"].mean()
ax.axvline(africa_avg, color="#e07b39", ls=":", lw=1.5, label=f"Africa average ({africa_avg:.2f})")
for bar, (_, row) in zip(bars, plot_df.iterrows()):
    ax.text(bar.get_width() + 0.03, bar.get_y() + bar.get_height() / 2,
            f"{row.lpi_overall:.2f} ({int(row.lpi_overall_year)})", va="center", fontsize=9)
ax.set_xlim(0, 4.6)
ax.set_xlabel("LPI overall score (latest edition; 2022 = 2023 LPI)")
ax.set_title("Logistics Performance Index: 15 African economies vs USA")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"Africa average LPI = {africa_avg:.2f}, i.e. {africa_avg / usa_lpi:.0%} of the US score")

## 2. Road infrastructure heatmap

Indicators are min-max scaled (0 = weakest, 1 = strongest within the 15 African countries) so that road density,
paving, LPI infrastructure and overall LPI can be compared on one grid. Raw values are annotated in each cell.
The USA has no road density / paving values in the World Bank extract and is therefore excluded.

In [ ]:
cols = {"road_density": "Road density\n(km/100 km²)", "paved_roads_pct": "Paved roads\n(%)",
        "lpi_infrastructure": "LPI\ninfrastructure", "lpi_overall": "LPI\noverall"}
africa = summary[summary["region"] == "Africa"].set_index("label")
heat_raw = africa[list(cols)].sort_values("lpi_infrastructure", ascending=False)
heat_scaled = (heat_raw - heat_raw.min()) / (heat_raw.max() - heat_raw.min())

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(heat_scaled.rename(columns=cols), annot=heat_raw.rename(columns=cols).round(1),
            fmt="", cmap="YlGnBu", linewidths=0.5, cbar_kws={"label": "Scaled score (0-1)"}, ax=ax)
ax.set_title("Road infrastructure profile of African trade-corridor economies")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 3. Trade openness vs road density

Does a denser road network correlate with higher trade intensity? Bubble size is merchandise exports (USD bn),
colour is the trade-openness category (Low < 40 %, Moderate 40-70 %, High > 70 % of GDP).

In [ ]:
scatter_df = summary[(summary["region"] == "Africa")].dropna(subset=["road_density", "trade_pct_gdp"]).copy()
scatter_df["exports_bn"] = scatter_df["merchandise_exports"] / 1e9
r = np.corrcoef(scatter_df["road_density"], scatter_df["trade_pct_gdp"])[0, 1]

fig = px.scatter(scatter_df, x="road_density", y="trade_pct_gdp", size="exports_bn", color="trade_openness_category",
                 text="label", size_max=45, trendline=None,
                 color_discrete_map={"Low": "#c0392b", "Moderate": "#f39c12", "High": "#27ae60"},
                 labels={"road_density": "Road density (km per 100 km² land, latest 2001-2010)",
                         "trade_pct_gdp": "Trade (% of GDP, latest 2024-2025)",
                         "trade_openness_category": "Openness"},
                 title=f"Road density vs trade openness (Pearson r = {r:.2f}, n = {len(scatter_df)})")
fig.update_traces(textposition="top center")
fig.update_layout(height=550)
fig.show()
print(f"Pearson r = {r:.3f}, r² = {r**2:.4f}")

## 4. LPI trends over time

LPI editions: 2012, 2014, 2016, 2018 and 2023 (stored as 2022). Not every country is surveyed in every edition, so
lines have gaps. The dashed navy line is the US benchmark.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
focus = ["ZAF", "EGY", "KEN", "NGA", "RWA", "GHA", "MAR", "ETH", "CMR"]
palette = sns.color_palette("tab10", len(focus))
for color, code in zip(palette, focus):
    d = lpi[lpi["country_code"] == code].sort_values("year")
    ax.plot(d["year"], d["lpi_overall"], marker="o", color=color, label=d["label"].iloc[0])
usa = lpi[lpi["country_code"] == "USA"].sort_values("year")
ax.plot(usa["year"], usa["lpi_overall"], ls="--", lw=2.5, color="#1f4e79", marker="s", label="USA")
africa_mean = lpi[lpi["region"] == "Africa"].groupby("year")["lpi_overall"].mean()
ax.plot(africa_mean.index, africa_mean.values, color="black", lw=3, alpha=0.4, label="Africa mean (surveyed)")
ax.set_xticks([2012, 2014, 2016, 2018, 2022])
ax.set_xticklabels(["2012", "2014", "2016", "2018", "2023*"])
ax.set_ylabel("LPI overall score")
ax.set_title("LPI trajectory by edition (*2023 edition stored by World Bank as 2022)")
ax.legend(ncol=2, fontsize=9, loc="lower left")
plt.tight_layout()
plt.show()

gap = (lpi[lpi["region"] == "Africa"].groupby("year")["lpi_overall"].mean()
       - lpi[lpi["country_code"] == "USA"].set_index("year")["lpi_overall"])
gap.rename("mean_africa_gap_vs_usa").to_frame()

## 5. Merchandise exports — Africa vs USA

Latest-year merchandise exports (current USD). The USA is shown separately because its exports are
roughly five times the combined total of all 15 African economies.

In [ ]:
latest_year = raw.loc[raw["indicator"] == "merchandise_exports", "year"].max()
exp = raw[(raw["indicator"] == "merchandise_exports") & (raw["year"] == latest_year)].copy()
exp["usd_bn"] = exp["value"] / 1e9
africa_exp = exp[exp["region"] == "Africa"].sort_values("usd_bn", ascending=False)
usa_exp = exp.loc[exp["country_code"] == "USA", "usd_bn"].iloc[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [3, 1]})
sns.barplot(data=africa_exp, x="usd_bn", y="label", hue="label", palette="crest", legend=False, ax=ax1)
for i, v in enumerate(africa_exp["usd_bn"]):
    ax1.text(v + 1, i, f"${v:,.1f}bn", va="center", fontsize=9)
ax1.set_xlabel(f"Merchandise exports {latest_year} (USD bn)")
ax1.set_ylabel("")
ax1.set_title(f"African merchandise exports, {latest_year}")

africa_total = africa_exp["usd_bn"].sum()
ax2.bar(["15 African\ncountries", "USA"], [africa_total, usa_exp], color=["#e07b39", "#1f4e79"])
for x, v in enumerate([africa_total, usa_exp]):
    ax2.text(x, v * 1.02, f"${v:,.0f}bn", ha="center")
ax2.set_title("Combined Africa vs USA")
ax2.set_ylabel("USD bn")
plt.tight_layout()
plt.show()

print(f"Africa total ${africa_total:,.1f}bn vs USA ${usa_exp:,.1f}bn -> USA/Africa = {usa_exp / africa_total:.1f}x")

## 6. Infrastructure drag: LPI infrastructure sub-score vs overall LPI

Points below the diagonal are countries whose physical infrastructure scores lower than their overall logistics
performance — i.e. infrastructure is the binding constraint on the corridor.

In [ ]:
latest_lpi = lpi.sort_values("year").groupby("country_code").tail(1).copy()
latest_lpi["infra_minus_overall"] = latest_lpi["lpi_infrastructure"] - latest_lpi["lpi_overall"]

fig, ax = plt.subplots(figsize=(8, 7))
sns.scatterplot(data=latest_lpi, x="lpi_overall", y="lpi_infrastructure", hue="region", s=120,
                palette={"Africa": "#e07b39", "North America": "#1f4e79"}, ax=ax)
for _, row in latest_lpi.iterrows():
    ax.annotate(row["label"], (row["lpi_overall"], row["lpi_infrastructure"]),
                textcoords="offset points", xytext=(5, 4), fontsize=8)
lims = [2.0, 4.1]
ax.plot(lims, lims, ls="--", color="grey", lw=1)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_title("Infrastructure sub-score vs overall LPI (latest edition)")
plt.tight_layout()
plt.show()

latest_lpi.sort_values("infra_minus_overall")[["label", "year", "lpi_overall", "lpi_infrastructure",
                                              "infra_minus_overall"]].head(6)

## 7. Key findings

1. **A persistent logistics gap.** The USA scores **3.80** on the 2023 LPI. South Africa (**3.70**) is the only African
   economy within 0.1 points; the 15-country African average of latest scores is **2.71 (≈71 % of the US level)**, and
   Cameroon trails by **-1.70**.
2. **Infrastructure is the binding constraint.** The average African LPI infrastructure sub-score is **2.54** vs **3.90**
   for the USA. Mozambique (-0.45), Uganda (-0.39), Kenya and Ethiopia (≈-0.26) have infrastructure scores well below
   their overall LPI.
3. **Most roads are unpaved.** Only Egypt (**92.2 %**) and Morocco (**70.4 %**) pave a majority of their network;
   the other 13 countries sit between **7.9 %** (Côte d'Ivoire) and **35.5 %** (Senegal).
4. **Road density alone does not explain trade openness.** Across 14 countries the correlation between road density and
   trade-%-of-GDP is essentially zero (r² ≈ 0.005). Port-oriented economies such as Mozambique (97 %) and Morocco (93 %)
   are highly open despite sparse networks — corridor *quality* and port connectivity matter more than raw density.
5. **Export scale is highly concentrated.** The 15 African economies exported about **$410bn** of merchandise in 2025,
   roughly one-fifth of US exports (**$2,185bn**). South Africa alone accounts for **28 %** of the African total; the top
   four (South Africa, Nigeria, Egypt, Morocco) account for **two-thirds**.
6. **Gradual convergence.** The mean gap between surveyed African countries and the USA narrowed from **-1.27** (2012) to
   **-1.00** (2023 edition), helped by a slight decline in the US score — though the 2023 edition surveyed only six of
   the 15 African countries in this panel.